In [1]:
import tkinter as tk
from tkinter import messagebox, ttk

class CPUScheduler:
    def __init__(self, root):
        self.root = root
        self.root.title("CPU Scheduling Analyzer")
        self.root.geometry("800x600")

        self.algorithms = ["FCFS", "SJF Non-Preemptive", "SJF Preemptive", "Priority Non-Preemptive", "Priority Preemptive", "Round Robin"]
        self.selected_algo = tk.StringVar()
        self.num_processes = tk.IntVar()
        self.time_quantum = tk.IntVar()

        self.setup_main_menu()

    def setup_main_menu(self):
        for widget in self.root.winfo_children():
            widget.destroy()

        tk.Label(self.root, text="Select Scheduling Algorithm", font=("Arial", 16)).pack(pady=10)
        algo_menu = ttk.Combobox(self.root, textvariable=self.selected_algo, values=self.algorithms, state='readonly', font=("Arial", 14))
        algo_menu.pack(pady=10)
        algo_menu.current(0)

        tk.Label(self.root, text="Enter Number of Processes:", font=("Arial", 14)).pack(pady=5)
        tk.Entry(self.root, textvariable=self.num_processes, font=("Arial", 14)).pack(pady=5)

        tk.Button(self.root, text="Next", command=self.setup_process_input, font=("Arial", 14), bg="green", fg="white").pack(pady=20)

    def setup_process_input(self):
        if self.num_processes.get() <= 0:
            messagebox.showerror("Error", "Number of processes should be more than 0")
            return

        for widget in self.root.winfo_children():
            widget.destroy()

        self.entries = []
        tk.Label(self.root, text="Enter Process Details", font=("Arial", 16)).pack(pady=10)
        columns = ["PID", "Arrival Time", "Burst Time"]

        if "Priority" in self.selected_algo.get():
            columns.append("Priority")

        frame = tk.Frame(self.root)
        frame.pack()

        for j, col in enumerate(columns):
            tk.Label(frame, text=col, font=("Arial", 12), width=15).grid(row=0, column=j)

        for i in range(self.num_processes.get()):
            row_entries = []
            tk.Label(frame, text=f"P{i+1}", font=("Arial", 12)).grid(row=i+1, column=0)
            for j in range(1, len(columns)):
                e = tk.Entry(frame, width=10)
                e.grid(row=i+1, column=j)
                row_entries.append(e)
            self.entries.append(row_entries)

        if "Round Robin" in self.selected_algo.get():
            tk.Label(self.root, text="Enter Time Quantum:", font=("Arial", 12)).pack(pady=5)
            tk.Entry(self.root, textvariable=self.time_quantum, font=("Arial", 12)).pack(pady=5)

        tk.Button(self.root, text="Show Results", command=self.compute_scheduling, font=("Arial", 14), bg="blue", fg="white").pack(pady=20)

    def compute_scheduling(self):
        try:
            process_data = []
            for i, row in enumerate(self.entries):
                arrival = int(row[0].get())
                burst = int(row[1].get())
                priority = int(row[2].get()) if len(row) == 3 else 0
                process_data.append({"pid": f"P{i+1}", "arrival": arrival, "burst": burst, "priority": priority})
        except:
            messagebox.showerror("Error", "Please fill all fields correctly.")
            return

        algo = self.selected_algo.get()
        tq = self.time_quantum.get() if "Round Robin" in algo else None

        if algo == "FCFS":
            result, gantt = self.fcfs(process_data)
        elif algo == "SJF Non-Preemptive":
            result, gantt = self.sjf_non_preemptive(process_data)
        elif algo == "SJF Preemptive":
            result, gantt = self.sjf_preemptive(process_data)
        elif algo == "Priority Non-Preemptive":
            result, gantt = self.priority_non_preemptive(process_data)
        elif algo == "Priority Preemptive":
            result, gantt = self.priority_preemptive(process_data)
        elif algo == "Round Robin":
            result, gantt = self.round_robin(process_data, tq)
        else:
            messagebox.showerror("Error", "Unknown Algorithm Selected")
            return

        self.display_result(result, gantt)

    def display_result(self, result, gantt):
        for widget in self.root.winfo_children():
            widget.destroy()

        tk.Label(self.root, text="Gantt Chart", font=("Arial", 16)).pack(pady=10)
        chart = tk.Canvas(self.root, height=50, bg="white")
        chart.pack(pady=10)

        start_x = 10
        for i, (pid, start, end) in enumerate(gantt):
            chart.create_rectangle(start_x, 10, start_x + (end - start)*20, 40, fill="lightblue")
            chart.create_text(start_x + (end - start)*10, 25, text=pid)
            chart.create_text(start_x, 45, text=str(start))
            start_x += (end - start)*20
        chart.create_text(start_x, 45, text=str(gantt[-1][2]))

        tk.Label(self.root, text="Process Table", font=("Arial", 14)).pack(pady=10)
        cols = ["PID", "Arrival", "Burst", "Priority", "Completion", "Turnaround", "Waiting"]
        tree = ttk.Treeview(self.root, columns=cols, show='headings')
        for col in cols:
            tree.heading(col, text=col)
            tree.column(col, width=90)
        for proc in result:
            tree.insert('', 'end', values=[proc.get("pid"), proc.get("arrival"), proc.get("burst"), proc.get("priority", "-"), proc.get("completion"), proc.get("turnaround"), proc.get("waiting")])
        tree.pack()

        avg_tat = sum([p['turnaround'] for p in result]) / len(result)
        avg_wt = sum([p['waiting'] for p in result]) / len(result)
        tk.Label(self.root, text=f"Average Turnaround Time: {avg_tat:.2f}", font=("Arial", 12)).pack(pady=5)
        tk.Label(self.root, text=f"Average Waiting Time: {avg_wt:.2f}", font=("Arial", 12)).pack(pady=5)

        tk.Button(self.root, text="Back to Main Menu", command=self.setup_main_menu, font=("Arial", 12)).pack(pady=10)

    # -------------------- Scheduling Algorithms -------------------------

    def fcfs(self, processes):
        processes.sort(key=lambda x: x['arrival'])
        time = 0
        gantt = []
        for p in processes:
            if time < p['arrival']:
                time = p['arrival']
            start = time
            time += p['burst']
            end = time
            p['completion'] = end
            p['turnaround'] = end - p['arrival']
            p['waiting'] = p['turnaround'] - p['burst']
            gantt.append((p['pid'], start, end))
        return processes, gantt

    def sjf_non_preemptive(self, processes):
        processes.sort(key=lambda x: (x['arrival'], x['burst']))
        completed = []
        time = 0
        gantt = []
        while processes:
            available = [p for p in processes if p['arrival'] <= time]
            if not available:
                time += 1
                continue
            current = min(available, key=lambda x: x['burst'])
            processes.remove(current)
            start = time
            time += current['burst']
            end = time
            current['completion'] = end
            current['turnaround'] = end - current['arrival']
            current['waiting'] = current['turnaround'] - current['burst']
            completed.append(current)
            gantt.append((current['pid'], start, end))
        return completed, gantt

    def sjf_preemptive(self, processes):
        n = len(processes)
        rem = [p['burst'] for p in processes]
        complete = 0
        time = 0
        shortest = -1
        minm = float('inf')
        check = False
        gantt = []
        last = -1
        while complete != n:
            for i in range(n):
                if (processes[i]['arrival'] <= time and rem[i] < minm and rem[i] > 0):
                    minm = rem[i]
                    shortest = i
                    check = True
            if not check:
                time += 1
                continue
            if last != shortest:
                gantt.append((processes[shortest]['pid'], time, time+1))
            else:
                gantt[-1] = (gantt[-1][0], gantt[-1][1], time+1)
            last = shortest
            rem[shortest] -= 1
            minm = rem[shortest] if rem[shortest] > 0 else float('inf')
            if rem[shortest] == 0:
                complete += 1
                finish_time = time + 1
                processes[shortest]['completion'] = finish_time
                processes[shortest]['turnaround'] = finish_time - processes[shortest]['arrival']
                processes[shortest]['waiting'] = processes[shortest]['turnaround'] - processes[shortest]['burst']
            time += 1
            check = False
        return processes, gantt

    def priority_non_preemptive(self, processes):
        processes.sort(key=lambda x: (x['arrival'], x['priority']))
        completed = []
        time = 0
        gantt = []
        while processes:
            available = [p for p in processes if p['arrival'] <= time]
            if not available:
                time += 1
                continue
            current = min(available, key=lambda x: x['priority'])
            processes.remove(current)
            start = time
            time += current['burst']
            end = time
            current['completion'] = end
            current['turnaround'] = end - current['arrival']
            current['waiting'] = current['turnaround'] - current['burst']
            completed.append(current)
            gantt.append((current['pid'], start, end))
        return completed, gantt

    def priority_preemptive(self, processes):
        n = len(processes)
        rem = [p['burst'] for p in processes]
        complete = 0
        time = 0
        gantt = []
        last = -1
        while complete != n:
            available = [(i, p) for i, p in enumerate(processes) if p['arrival'] <= time and rem[i] > 0]
            if not available:
                time += 1
                continue
            shortest = min(available, key=lambda x: x[1]['priority'])[0]
            if last != shortest:
                gantt.append((processes[shortest]['pid'], time, time+1))
            else:
                gantt[-1] = (gantt[-1][0], gantt[-1][1], time+1)
            last = shortest
            rem[shortest] -= 1
            if rem[shortest] == 0:
                complete += 1
                finish_time = time + 1
                processes[shortest]['completion'] = finish_time
                processes[shortest]['turnaround'] = finish_time - processes[shortest]['arrival']
                processes[shortest]['waiting'] = processes[shortest]['turnaround'] - processes[shortest]['burst']
            time += 1
        return processes, gantt

    def round_robin(self, processes, tq):
        from collections import deque
        queue = deque()
        time = 0
        processes.sort(key=lambda x: x['arrival'])
        rem = {p['pid']: p['burst'] for p in processes}
        gantt = []
        arrival_map = {p['pid']: p['arrival'] for p in processes}
        pid_map = {p['pid']: p for p in processes}
        queue.extend([p for p in processes if p['arrival'] == 0])
        seen = set(p['pid'] for p in queue)

        while queue or any(rem.values()):
            if queue:
                current = queue.popleft()
                p = pid_map[current['pid']]
                bt = min(rem[p['pid']], tq)
                gantt.append((p['pid'], time, time+bt))
                time += bt
                rem[p['pid']] -= bt
                for proc in processes:
                    if proc['arrival'] <= time and proc['pid'] not in seen and rem[proc['pid']] > 0:
                        queue.append(proc)
                        seen.add(proc['pid'])
                if rem[p['pid']] > 0:
                    queue.append(p)
                else:
                    p['completion'] = time
                    p['turnaround'] = time - p['arrival']
                    p['waiting'] = p['turnaround'] - p['burst']
            else:
                time += 1
                for proc in processes:
                    if proc['arrival'] <= time and proc['pid'] not in seen and rem[proc['pid']] > 0:
                        queue.append(proc)
                        seen.add(proc['pid'])
        return processes, gantt

if __name__ == '__main__':
    root = tk.Tk()
    app = CPUScheduler(root)
    root.mainloop()
